                # 01 Data Foundation Lab

                Purpose: pull latest code, run Phase 1 data foundation stages,
                create or refresh OHLCV data, validate data quality, record
                futures-metrics and optional aggTrades status, and inspect the
                resulting data-quality report.
                


## 1. Mount Drive And Project Root


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)


## 2. Pull Latest Code


In [ ]:
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/umutergul74/btcusdt_quant_research.git"
GITHUB_BRANCH = "main"
project_path = Path(PROJECT_ROOT)

if (project_path / ".git").exists():
    subprocess.run(["git", "-C", PROJECT_ROOT, "remote", "set-url", "origin", GITHUB_REPO_URL], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)
else:
    visible_items = [p.name for p in project_path.iterdir() if not p.name.startswith(".")]
    if visible_items:
        raise RuntimeError(
            f"{PROJECT_ROOT} exists but is not a git checkout. "
            "Move existing Drive artifacts aside or clone the repository there before continuing. "
            f"Visible items: {visible_items[:10]}"
        )
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, PROJECT_ROOT], check=True)

if f"{PROJECT_ROOT}/src" not in sys.path:
    sys.path.append(f"{PROJECT_ROOT}/src")
print("Repository is ready at:", PROJECT_ROOT)


## 3. Install Dependencies And Stage Helper


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

from pprint import pprint
from btc_quant.pipeline.stages import run_stage
from btc_quant.pipeline.stage_registry import list_stage_specs

RUN_ID = "debug_colab"

def run_stage_checked(stage_name, config_name=None):
    print(f"\n=== {stage_name} ===")
    result = run_stage(stage_name, config_name=config_name, project_root=PROJECT_ROOT, run_id=RUN_ID)
    pprint(result)
    if result.get("status") == "FAIL":
        raise RuntimeError(f"Stage failed: {stage_name}")
    return result


## 4. Review Data Configuration


In [ ]:
                import yaml
                from pathlib import Path

                data_config_path = Path(PROJECT_ROOT) / "configs" / "data.yaml"
                print(data_config_path)
                print(data_config_path.read_text())
                


## 5. Run OHLCV Foundation Stage


In [ ]:
ohlcv_result = run_stage_checked("data_ohlcv", config_name="data")


## 6. Run Data Quality Checks


In [ ]:
quality_result = run_stage_checked("data_quality", config_name="data")


## 7. Record Futures Metrics Status


In [ ]:
futures_result = run_stage_checked("futures_metrics", config_name="data")


## 8. Record Optional aggTrades Status


In [ ]:
aggtrades_result = run_stage_checked("aggtrades_optional", config_name="data")


## 9. Inspect Saved Data Quality Report


In [ ]:
                import json
                from pathlib import Path

                report_path = Path(PROJECT_ROOT) / "reports" / "experiments" / RUN_ID / "data_quality_summary.json"
                print(report_path)
                pprint(json.loads(report_path.read_text()))
                


## 10. Confirm Phase Boundary


In [ ]:
                print("Phase 1 data foundation complete.")
                print("No formal backtest, execution simulator, position sizing, or dashboard stage was run.")
                


In [ ]:
# Auto-close the Colab runtime after every previous cell has completed.
# Set this to False before running if you want to keep the session alive for inspection.
AUTO_CLOSE_COLAB_RUNTIME = True

if AUTO_CLOSE_COLAB_RUNTIME:
    try:
        from google.colab import runtime
        print("All notebook cells completed. Releasing the Colab runtime now.")
        runtime.unassign()
    except Exception as exc:
        print(f"Auto-close skipped outside Colab or because runtime release failed: {exc}")
